# Building a QA Agent with Claude on Vertex AI

Building a basic Question Answering (QA) agent using Anthropic's Claude model on Vertex AI.

## Import Libraries and Setup

In [1]:
import base64
from pathlib import Path

from IPython.display import Markdown, display
from anthropic import AnthropicVertex
from anthropic.types import (
    Base64PDFSourceParam,
    DocumentBlockParam,
    MessageParam,
    TextBlockParam,
)
from dotenv import load_dotenv

from helpers import authenticate

In [2]:
_ = load_dotenv()

## Initialize the Client and Load Data

In [3]:
credentials, project_id = authenticate()

In [4]:
client = AnthropicVertex(
    project_id=project_id,
    region="global",
    access_token=credentials.token,
)

Reading the insurance policy PDF (`2026AnthemgHIPSBC.pdf`) and encode it in base64 so it can be passed to the model as context.

In [5]:
with Path("../data/2026AnthemgHIPSBC.pdf").open("rb") as file:
    pdf_data = base64.standard_b64encode(file.read()).decode("utf-8")

## Query the Model

In [6]:
prompt = "How much would I pay for mental health therapy?"

In [7]:
response = client.messages.create(
    model="claude-haiku-4-5@20251001",
    max_tokens=1024,
    system="""You are an expert insurance agent designed to assist with 
    coverage queries. Use the provided documents to answer questions 
    about insurance policies. If the information is not available in 
    the documents, respond with 'I don't know'""",
    messages=[
        MessageParam(
            role="user",
            content=[
                DocumentBlockParam(
                    type="document",
                    source=Base64PDFSourceParam(
                        type="base64",
                        media_type="application/pdf",
                        data=pdf_data,
                    ),
                ),
                TextBlockParam(
                    type="text",
                    text=prompt,
                ),
            ],
        )
    ],
)

In [8]:
response_text = response.content[0].text.replace("$", r"\\$")
display(Markdown(response_text))

# Mental Health Therapy Costs

Based on the plan document, here are the costs for mental health outpatient services (which would include therapy):

**In-Network Provider:**
- 10% coinsurance

**Out-of-Network Provider:**
- 30% coinsurance

## Important Notes:

1. **After Deductible**: These coinsurance amounts apply *after* you've met your deductible:
   - \\$1,700 individual / \\$3,400 family (in-network)
   - \\$3,400 individual / \\$6,800 family (out-of-network)

2. **No Referral Required**: You can see a mental health provider of your choice without needing a referral from your primary care doctor.

3. **Out-of-Pocket Limit**: Your costs count toward your annual out-of-pocket limit:
   - \\$2,600 individual / \\$5,200 family (in-network)
   - \\$5,200 individual / \\$10,400 family (out-of-network)

For specific questions about your individual situation or to find in-network mental health providers, you can contact the plan at **(855) 431-5540** or visit **includedhealth.com/google**.

## Refactor into an Agent Class

In [9]:
%%writefile agents.py
import base64
from pathlib import Path

from anthropic import AnthropicVertex
from anthropic.types import (
    Base64PDFSourceParam,
    DocumentBlockParam,
    MessageParam,
    TextBlockParam,
)
from dotenv import load_dotenv

from helpers import authenticate

class PolicyAgent:
    def __init__(self) -> None:
        load_dotenv()
        credentials, project_id = authenticate()
        self.client = AnthropicVertex(
            project_id=project_id,
            region="global",
            access_token=credentials.token,
        )
        with Path("../data/2026AnthemgHIPSBC.pdf").open("rb") as file:
            self.pdf_data = base64.standard_b64encode(file.read()).decode("utf-8")

    def answer_query(self, prompt: str) -> str:
        response = self.client.messages.create(
            model="claude-haiku-4-5@20251001",
            max_tokens=1024,
            system="You are an expert insurance agent designed to assist with coverage queries. Use the provided documents to answer questions about insurance policies. If the information is not available in the documents, respond with 'I don't know'",
            messages=[
                MessageParam(
                    role="user",
                    content=[
                        DocumentBlockParam(
                            type="document",
                            source=Base64PDFSourceParam(
                                type="base64",
                                media_type="application/pdf",
                                data=self.pdf_data,
                            ),
                        ),
                        TextBlockParam(
                            type="text",
                            text=prompt,
                        ),
                    ],
                )
            ],
        )
        
        return response.content[0].text.replace("$", r"\\$")

Overwriting agents.py


## Test the Agent Class

In [10]:
from agents import PolicyAgent

print("Running Health Insurance Policy Agent")
agent = PolicyAgent()
prompt = "How much would I pay for mental health therapy?"

response = agent.answer_query(prompt)
display(Markdown(response))

Running Health Insurance Policy Agent


Based on the Summary of Benefits and Coverage document for the Anthem Blue Cross gHIP HDHP plan, here are the costs for mental health therapy (outpatient services):

**In-Network Provider:**
- Office Visit: 10% coinsurance
- Other Outpatient Services: 10% coinsurance

**Out-of-Network Provider:**
- Office Visit: 30% coinsurance
- Other Outpatient Services: 30% coinsurance

**Important notes:**
- These coinsurance amounts apply **after your deductible has been met**
- The plan's deductible is \\$1,700 for an individual or \\$3,400 for a family (in-network)
- Once you reach your out-of-pocket limit (\\$2,600 for individuals or \\$5,200 for families in-network), the plan covers 100% of remaining covered services

So your actual out-of-pocket cost would depend on whether you've met your deductible and out-of-pocket limits for the year.